In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_PATH = Path(
    "/Users/jasminetaha/Documents/Retroverse Analytics"
)

CLEANED_DATA = BASE_PATH / "data" / "cleaned"
RAW_DATA = BASE_PATH / "data" / "raw"

In [2]:
sales_orders = pd.read_csv(
    CLEANED_DATA / "sales_orders.csv"
)

customer_priority = pd.read_csv(
    CLEANED_DATA / "customer_priority_summary.csv"
)

customer_propensity = pd.read_csv(
    CLEANED_DATA / "customer_propensity.csv"
)

channels = pd.read_csv(
    RAW_DATA / "shopify_channel_performance.csv"
)

sales_orders["Created at"] = pd.to_datetime(
    sales_orders["Created at"],
    errors="coerce",
    utc=True
)

print("Sales orders:", len(sales_orders))
print("Customer priorities:", len(customer_priority))
print("Customer records:", len(customer_propensity))
print("Channel rows:", len(channels))

Sales orders: 118
Customer priorities: 3
Customer records: 93
Channel rows: 50


In [3]:
channel_summary = (
    channels
    .groupby("Channel", dropna=False)
    .agg(
        Sessions=("Sessions", "sum"),
        Sales=("Sales", "sum"),
        Orders=("Orders", "sum"),
        New_Customers=("Orders from new customers", "sum"),
        Returning_Customers=("Orders from returning customers", "sum")
    )
    .reset_index()
)

channel_summary["Conversion_Rate_%"] = (
    channel_summary["Orders"]
    / channel_summary["Sessions"]
    * 100
)

channel_summary["New_Customer_%"] = (
    channel_summary["New_Customers"]
    / channel_summary["Orders"]
    * 100
)

channel_summary[
    [
        "Channel",
        "Sessions",
        "Orders",
        "Conversion_Rate_%",
        "New_Customers",
        "Returning_Customers",
        "New_Customer_%"
    ]
].sort_values("Orders", ascending=False).round(2)

,Channel,Sessions,Orders,Conversion_Rate_%,New_Customers,Returning_Customers,New_Customer_%
8,instagram,3429,62.0,1.81,53.0,9.0,85.48
4,direct,3678,56.0,1.52,15.0,41.0,26.79
7,google,644,23.0,3.57,20.0,3.0,86.96
13,unattributed,831,15.0,1.81,14.0,1.0,93.33
0,baidu,4,0.0,0.00,0.0,0.0,NaN
1,bing,13,0.0,0.00,0.0,0.0,NaN
2,brave,6,0.0,0.00,0.0,0.0,NaN
3,chatgpt.com,8,0.0,0.00,0.0,0.0,NaN
5,facebook,110,0.0,0.00,0.0,0.0,NaN
6,gmail,5,0.0,0.00,0.0,0.0,NaN


### Marketing Evidence

Instagram brings the most customers, while Google has the strongest conversion rate.

Most Instagram and Google orders are from new customers.

Direct traffic is mainly returning customers, which makes it more useful as a retention signal than a new customer acquisition channel.

In [4]:
marketing_options = pd.DataFrame({
    "Strategy": [
        "Meta Prospecting",
        "Meta Retargeting",
        "Google / SEO",
        "Customer Reactivation"
    ],

    "Purpose": [
        "Reach new customers",
        "Bring interested visitors back",
        "Capture people already searching",
        "Bring previous customers back"
    ],

    "Historical_Evidence": [
        "Instagram: 3429 sessions, 62 orders",
        "Instagram already brings strong traffic",
        "Google: 644 sessions, 23 orders",
        "10 medium/high priority previous customers"
    ],

    "Evidence_Strength": [
        "Strong audience evidence / no paid history",
        "Moderate",
        "Strong",
        "Strong customer evidence"
    ]
})

marketing_options

,Strategy,Purpose,Historical_Evidence,Evidence_Strength
0,Meta Prospecting,Reach new customers,"Instagram: 3429 sessions, 62 orders",Strong audience evidence / no paid history
1,Meta Retargeting,Bring interested visitors back,Instagram already brings strong traffic,Moderate
2,Google / SEO,Capture people already searching,"Google: 644 sessions, 23 orders",Strong
3,Customer Reactivation,Bring previous customers back,10 medium/high priority previous customers,Strong customer evidence


### Marketing Opportunities

Instagram is the strongest channel for reaching new customers, while Google has the highest conversion rate.

There is also a large group of inactive previous customers that could be brought back.

Since Retroverse has never used Meta Ads before, paid ads will be treated as a new test rather than assuming they will perform the same as organic Instagram.


Meta Prospecting means showing paid Instagram/Facebook ads to new people who don’t already know Retroverse. This is our biggest new-customer experiment because Instagram already brought 62 attributed orders organically.

Meta Retargeting means ads to people who already interacted with Retroverse — for example, someone who visited the website or interacted with the brand but didn’t buy. They’re warmer than completely new people.

Google / SEO (Search Engine Optimization) is different. Google already has a 3.57% conversion rate, the highest major channel in the dataset. That tells us people who find Retroverse through Google appear to have strong buying intent. We can later decide whether the 30K should actually include paid Google or primarily support organic search — we shouldn’t automatically spend money there just because organic Google performs well.

Customer Reactivation targets previous buyers. We now know exactly why this matters: 89 customers are inactive, including 19 high-value inactive customers and 5 repeat customers at risk, while our propensity ranking gives us 10 particularly strong reactivation targets.

In [5]:
orders = pd.read_csv(
    RAW_DATA / "shopify_orders.csv"
)

orders["Created at"] = pd.to_datetime(
    orders["Created at"],
    errors="coerce",
    utc=True
)

# Keep line items belonging to our validated sales
valid_ids = set(sales_orders["Id"])

recent_items = orders[
    orders["Id"].isin(valid_ids)
    & orders["Created at"].dt.year.isin([2025, 2026])
].copy()

recent_items["Lineitem quantity"] = pd.to_numeric(
    recent_items["Lineitem quantity"],
    errors="coerce"
)

recent_items["Lineitem price"] = pd.to_numeric(
    recent_items["Lineitem price"],
    errors="coerce"
)

In [6]:
def assign_unit_cogs(name):
    name = str(name).lower()

    if "smoking bills" in name:
        return 700

    elif "vinyl" in name:
        return 90

    elif "signature" in name:
        return 350

    else:
        return 355


recent_items["Unit_COGS"] = (
    recent_items["Lineitem name"]
    .apply(assign_unit_cogs)
)

recent_items["Total_COGS"] = (
    recent_items["Lineitem quantity"]
    * recent_items["Unit_COGS"]
)

In [8]:
recent_cogs = recent_items["Total_COGS"].sum()

recent_revenue = sales_orders[
    sales_orders["Created at"].dt.year.isin([2025, 2026])
]["Net Revenue"].sum()

recent_orders = sales_orders[
    sales_orders["Created at"].dt.year.isin([2025, 2026])
]["Id"].nunique()

gross_profit = recent_revenue - recent_cogs

unit_economics = pd.DataFrame({
    "Metric": [
        "Recent Revenue",
        "Recent Orders",
        "Total COGS",
        "Gross Profit",
        "Average Revenue per Order",
        "Average Gross Profit per Order"
    ],
    "Value": [
        recent_revenue,
        recent_orders,
        recent_cogs,
        gross_profit,
        recent_revenue / recent_orders,
        gross_profit / recent_orders
    ]
})

unit_economics.round(2)

,Metric,Value
0,Recent Revenue,45620.00
1,Recent Orders,18.00
2,Total COGS,8780.00
3,Gross Profit,36840.00
4,Average Revenue per Order,2534.44
5,Average Gross Profit per Order,2046.67


In [10]:
normal_sales = sales_orders[
    sales_orders["Net Revenue"] < 19000
].copy()

normal_recent_sales = normal_sales[
    normal_sales["Created at"].dt.year.isin([2025, 2026])
]

normal_ids = set(normal_recent_sales["Id"])

normal_recent_items = recent_items[
    recent_items["Id"].isin(normal_ids)
]

normal_revenue = normal_recent_sales["Net Revenue"].sum()
normal_orders = normal_recent_sales["Id"].nunique()
normal_cogs = normal_recent_items["Total_COGS"].sum()
normal_gross_profit = normal_revenue - normal_cogs

normal_economics = pd.DataFrame({
    "Metric": [
        "Revenue",
        "Orders",
        "COGS",
        "Gross Profit",
        "Average Revenue per Order",
        "Average Gross Profit per Order",
        "Gross Margin %"
    ],
    "Value": [
        normal_revenue,
        normal_orders,
        normal_cogs,
        normal_gross_profit,
        normal_revenue / normal_orders,
        normal_gross_profit / normal_orders,
        normal_gross_profit / normal_revenue * 100
    ]
})

normal_economics.round(2)

,Metric,Value
0,Revenue,26620.00
1,Orders,17.00
2,COGS,5585.00
3,Gross Profit,21035.00
4,Average Revenue per Order,1565.88
5,Average Gross Profit per Order,1237.35
6,Gross Margin %,79.02


### Marketing Profitability

Recent orders have strong gross margins, but the 19,000 EGP for ahmed miltry order increases the average.

For the marketing model, I removed this order when estimating normal revenue and profit per order so the expected results are more realistic.


In [11]:
meta_test = pd.DataFrame({
    "Scenario": [
        "Conservative",
        "Base",
        "Optimistic"
    ],
    "CPA": [
        1000,
        700,
        450
    ]
})

meta_test["Gross_Profit_Per_Order"] = 1237.35

meta_test["Profit_After_Ads_Per_Order"] = (
    meta_test["Gross_Profit_Per_Order"]
    - meta_test["CPA"]
)

meta_test["Break_Even"] = (
    meta_test["Profit_After_Ads_Per_Order"] >= 0
)

meta_test.round(2)

,Scenario,CPA,Gross_Profit_Per_Order,Profit_After_Ads_Per_Order,Break_Even
0,Conservative,1000,1237.35,237.35,True
1,Base,700,1237.35,537.35,True
2,Optimistic,450,1237.35,787.35,True


### Meta Ads Assumptions

Retroverse has never run Meta Ads, so there is no historical Cost per Acquisition (CPA) to use.

I tested different CPA levels instead of assuming one result. Based on recent normal orders, the average gross profit per order is around 1,237 EGP before advertising and other expenses.

In [12]:
meta_budgets = [5000, 10000, 15000, 20000]

simulation = []

for _, scenario in meta_test.iterrows():
    for budget in meta_budgets:

        expected_orders = budget / scenario["CPA"]

        expected_revenue = (
            expected_orders * 1565.88
        )

        expected_gross_profit = (
            expected_orders * 1237.35
        )

        profit_after_ads = (
            expected_gross_profit - budget
        )

        roas = expected_revenue / budget

        simulation.append({
            "Scenario": scenario["Scenario"],
            "Ad_Spend": budget,
            "CPA": scenario["CPA"],
            "Expected_Orders": expected_orders,
            "Expected_Revenue": expected_revenue,
            "Gross_Profit_Before_Ads": expected_gross_profit,
            "Profit_After_Ads": profit_after_ads,
            "ROAS": roas
        })

meta_simulation = pd.DataFrame(simulation)

meta_simulation.round(2)

,Scenario,Ad_Spend,CPA,Expected_Orders,Expected_Revenue,Gross_Profit_Before_Ads,Profit_After_Ads,ROAS
0,Conservative,5000,1000,5.00,7829.40,6186.75,1186.75,1.57
1,Conservative,10000,1000,10.00,15658.80,12373.50,2373.50,1.57
2,Conservative,15000,1000,15.00,23488.20,18560.25,3560.25,1.57
3,Conservative,20000,1000,20.00,31317.60,24747.00,4747.00,1.57
4,Base,5000,700,7.14,11184.86,8838.21,3838.21,2.24
5,Base,10000,700,14.29,22369.71,17676.43,7676.43,2.24
6,Base,15000,700,21.43,33554.57,26514.64,11514.64,2.24
7,Base,20000,700,28.57,44739.43,35352.86,15352.86,2.24
8,Optimistic,5000,450,11.11,17398.67,13748.33,8748.33,3.48
9,Optimistic,10000,450,22.22,34797.33,27496.67,17496.67,3.48


### Meta Ads Simulation

Meta Ads could be profitable if the Cost per Acquisition stays below the profit generated by an order.

In the base scenario, a 10,000 EGP ad budget could generate around 14 orders and 22,370 EGP in revenue.

These are estimated scenarios because Retroverse has never run paid Meta Ads before. Actual performance should be tested with a smaller budget before increasing ad spend.

In [13]:
reactivation_targets = customer_propensity[
    customer_propensity["Priority"].isin(["High", "Medium"])
].copy()

reactivation_summary = pd.DataFrame({
    "Metric": [
        "Target Customers",
        "Previous Revenue",
        "Average Previous Spend",
        "Repeat Customers in Target Group"
    ],
    "Value": [
        len(reactivation_targets),
        reactivation_targets["Monetary"].sum(),
        reactivation_targets["Monetary"].mean(),
        (reactivation_targets["Frequency"] >= 2).sum()
    ]
})

reactivation_summary.round(2)

,Metric,Value
0,Target Customers,10.0
1,Previous Revenue,45960.0
2,Average Previous Spend,4596.0
3,Repeat Customers in Target Group,6.0


In [14]:
reactivation_scenarios = pd.DataFrame({
    "Scenario": [
        "Conservative",
        "Base",
        "Optimistic"
    ],
    "Return_Rate_%": [
        10,
        20,
        30
    ]
})

reactivation_scenarios["Target_Customers"] = len(
    reactivation_targets
)

reactivation_scenarios["Expected_Returning_Customers"] = (
    reactivation_scenarios["Target_Customers"]
    * reactivation_scenarios["Return_Rate_%"]
    / 100
)

# Use normal historical AOV, not the inflated target-group average
reactivation_scenarios["Expected_Revenue"] = (
    reactivation_scenarios["Expected_Returning_Customers"]
    * 1565.88
)

reactivation_scenarios.round(2)

,Scenario,Return_Rate_%,Target_Customers,Expected_Returning_Customers,Expected_Revenue
0,Conservative,10,10,1.0,1565.88
1,Base,20,10,2.0,3131.76
2,Optimistic,30,10,3.0,4697.64


### Customer Reactivation

There are only 10 high and medium priority customers, so reactivation does not need a large budget.

In the base scenario, bringing back 2 of these customers could generate around 3,132 EGP in revenue.

This should be a small part of the marketing budget while most of the money focuses on reaching new customers.

### Google Opportunity

Google has the highest historical conversion rate at 3.57%, showing that people who find Retroverse through search have strong buying intent.

This does not prove that Google Ads would perform the same way, so the historical conversion rate will be used as evidence of search demand rather than expected paid ad performance.

In [16]:
initial_budget = pd.DataFrame({
    "Strategy": [
        "Meta Prospecting",
        "Meta Retargeting",
        "Google / SEO",
        "Customer Reactivation"
    ],
    "Budget_EGP": [
        18000,
        5000,
        6000,
        1000
    ]
})

initial_budget["Budget_%"] = (
    initial_budget["Budget_EGP"]
    / initial_budget["Budget_EGP"].sum()
    * 100
)

initial_budget

,Strategy,Budget_EGP,Budget_%
0,Meta Prospecting,18000,60.000000
1,Meta Retargeting,5000,16.666667
2,Google / SEO,6000,20.000000
3,Customer Reactivation,1000,3.333333


In [17]:
meta_prospecting_budget = 18000

meta_budget_test = meta_test.copy()

meta_budget_test["Ad_Spend"] = meta_prospecting_budget

meta_budget_test["Expected_Orders"] = (
    meta_budget_test["Ad_Spend"]
    / meta_budget_test["CPA"]
)

meta_budget_test["Expected_Revenue"] = (
    meta_budget_test["Expected_Orders"]
    * 1565.88
)

meta_budget_test["Gross_Profit_Before_Ads"] = (
    meta_budget_test["Expected_Orders"]
    * 1237.35
)

meta_budget_test["Profit_After_Ads"] = (
    meta_budget_test["Gross_Profit_Before_Ads"]
    - meta_budget_test["Ad_Spend"]
)

meta_budget_test["ROAS"] = (
    meta_budget_test["Expected_Revenue"]
    / meta_budget_test["Ad_Spend"]
)

meta_budget_test.round(2)

,Scenario,CPA,Gross_Profit_Per_Order,Profit_After_Ads_Per_Order,Break_Even,Ad_Spend,Expected_Orders,Expected_Revenue,Gross_Profit_Before_Ads,Profit_After_Ads,ROAS
0,Conservative,1000,1237.35,237.35,True,18000,18.00,28185.84,22272.30,4272.30,1.57
1,Base,700,1237.35,537.35,True,18000,25.71,40265.49,31817.57,13817.57,2.24
2,Optimistic,450,1237.35,787.35,True,18000,40.00,62635.20,49494.00,31494.00,3.48


In [18]:
gross_profit_per_order = 1237.35

break_even_cpa = gross_profit_per_order

target_cpa_20_margin = (
    gross_profit_per_order * 0.80
)

target_cpa_40_margin = (
    gross_profit_per_order * 0.60
)

print(
    "Break-even CPA:",
    round(break_even_cpa, 2),
    "EGP"
)

print(
    "CPA leaving 20% of gross profit:",
    round(target_cpa_20_margin, 2),
    "EGP"
)

print(
    "CPA leaving 40% of gross profit:",
    round(target_cpa_40_margin, 2),
    "EGP"
)

Break-even CPA: 1237.35 EGP
CPA leaving 20% of gross profit: 989.88 EGP
CPA leaving 40% of gross profit: 742.41 EGP


In [19]:
meta_rules = pd.DataFrame({
    "CPA_Range": [
        "Below 750 EGP",
        "750 - 1000 EGP",
        "1000 - 1237 EGP",
        "Above 1237 EGP"
    ],
    "Action": [
        "Scale",
        "Continue testing",
        "Reduce spend and optimize",
        "Pause"
    ]
})

meta_rules

,CPA_Range,Action
0,Below 750 EGP,Scale
1,750 - 1000 EGP,Continue testing
2,1000 - 1237 EGP,Reduce spend and optimize
3,Above 1237 EGP,Pause


### Meta Ads Decision

Meta Ads should not receive the full budget immediately because Retroverse has no previous paid advertising data.

The campaign should start as a test. If the Cost per Acquisition stays below around 750 EGP, the budget can be increased.

If it goes above the break-even level of around 1,237 EGP, the campaign should be paused or changed.

### CPA Target

Based on recent normal orders, around 1,237 EGP is the gross-profit break-even Cost per Acquisition.

I would aim to keep the Cost per Acquisition below around 750 EGP so there is still enough profit left after production and advertising costs.

In [20]:
target_strategy = pd.DataFrame({
    "Area": [
        "Location",
        "Customer",
        "Acquisition",
        "Main Product",
        "Test Product",
        "Reactivation"
    ],

    "Target": [
        "Cairo, New Cairo, Sheikh Zayed / October",
        "New customers",
        "Instagram / Meta",
        "Custom Orders",
        "Custom Vinyl",
        "High and medium priority previous customers"
    ],

    "Reason": [
        "Strongest order and revenue areas",
        "Repeat purchasing is currently very low",
        "Instagram historically brought the most new customers",
        "Highest historical demand with strong margins",
        "85% margin with potential to grow",
        "Best previous customers to try bringing back"
    ]
})

target_strategy

,Area,Target,Reason
0,Location,"Cairo, New Cairo, Sheikh Zayed / October",Strongest order and revenue areas
1,Customer,New customers,Repeat purchasing is currently very low
2,Acquisition,Instagram / Meta,Instagram historically brought the most new cu...
3,Main Product,Custom Orders,Highest historical demand with strong margins
4,Test Product,Custom Vinyl,85% margin with potential to grow
5,Reactivation,High and medium priority previous customers,Best previous customers to try bringing back


### Marketing Target

The main target should be new customers in Greater Cairo, especially Cairo, New Cairo and Sheikh Zayed / October.

Instagram is the strongest channel for reaching new customers, and Custom Orders should be the main product advertised because they have the strongest combination of demand and profit.

Custom Vinyl can be tested with a smaller budget because the margin is high but there is less sales history.

In [21]:
meta_rollout = pd.DataFrame({
    "Stage": [
        "Test",
        "Scale 1",
        "Scale 2"
    ],
    "Budget_EGP": [
        5000,
        6000,
        7000
    ],
    "Rule": [
        "Launch and measure real CPA",
        "Release if CPA is acceptable",
        "Release if performance stays profitable"
    ]
})

meta_rollout["Cumulative_Budget"] = (
    meta_rollout["Budget_EGP"].cumsum()
)

meta_rollout

,Stage,Budget_EGP,Rule,Cumulative_Budget
0,Test,5000,Launch and measure real CPA,5000
1,Scale 1,6000,Release if CPA is acceptable,11000
2,Scale 2,7000,Release if performance stays profitable,18000


So 18K is reserved, but only 5K gets risked initially.

If the first 5K produces a CPA around 700 EGP or below, we have evidence to scale. If CPA starts approaching 1,000 EGP, we’re cautious. If it goes beyond the ~1,237 EGP gross-profit break-even CPA, we don’t blindly release the rest.



In [22]:
channel_plan = pd.DataFrame({
    "Strategy": [
        "Meta Prospecting",
        "Meta Retargeting",
        "Google / SEO",
        "Customer Reactivation"
    ],

    "Historical_Evidence": [
        "Instagram: 3429 sessions and 62 orders",
        "Instagram traffic creates a possible warm audience",
        "Google: 644 sessions and 23 orders",
        "10 medium/high priority customers"
    ],

    "Known_Performance": [
        "Organic Instagram only",
        "No paid retargeting history",
        "Organic Google only",
        "Historical customer behavior"
    ],

    "Main_Risk": [
        "No historical paid CPA",
        "Audience may be too small",
        "No historical Google Ads data",
        "Only 10 priority customers"
    ]
})

channel_plan

,Strategy,Historical_Evidence,Known_Performance,Main_Risk
0,Meta Prospecting,Instagram: 3429 sessions and 62 orders,Organic Instagram only,No historical paid CPA
1,Meta Retargeting,Instagram traffic creates a possible warm audi...,No paid retargeting history,Audience may be too small
2,Google / SEO,Google: 644 sessions and 23 orders,Organic Google only,No historical Google Ads data
3,Customer Reactivation,10 medium/high priority customers,Historical customer behavior,Only 10 priority customers


In [23]:
recommended_budget = pd.DataFrame({
    "Strategy": [
        "Meta Prospecting",
        "Meta Retargeting",
        "Google Search Test",
        "Customer Reactivation",
        "Reserve / Scale Best Channel"
    ],

    "Budget_EGP": [
        15000,
        3000,
        5000,
        1000,
        6000
    ]
})

recommended_budget["Budget_%"] = (
    recommended_budget["Budget_EGP"]
    / recommended_budget["Budget_EGP"].sum()
    * 100
)

recommended_budget

,Strategy,Budget_EGP,Budget_%
0,Meta Prospecting,15000,50.000000
1,Meta Retargeting,3000,10.000000
2,Google Search Test,5000,16.666667
3,Customer Reactivation,1000,3.333333
4,Reserve / Scale Best Channel,6000,20.000000


### 30,000 EGP Marketing Budget

I would not spend the full 30,000 EGP immediately.

The largest part should go toward acquiring new customers through Meta because Instagram has historically been Retroverse's strongest acquisition channel.

Google should also be tested because its historical traffic had the highest conversion rate, while smaller amounts can be used for retargeting and customer reactivation.

I would keep 6,000 EGP in reserve and move it toward whichever channel performs best after the first tests.

In [24]:
google_test = pd.DataFrame({
    "Scenario": [
        "Conservative",
        "Base",
        "Optimistic"
    ],
    "CPA": [
        1100,
        800,
        550
    ]
})

google_budget = 5000

google_test["Ad_Spend"] = google_budget

google_test["Expected_Orders"] = (
    google_test["Ad_Spend"]
    / google_test["CPA"]
)

google_test["Expected_Revenue"] = (
    google_test["Expected_Orders"]
    * 1565.88
)

google_test["Gross_Profit_Before_Ads"] = (
    google_test["Expected_Orders"]
    * 1237.35
)

google_test["Profit_After_Ads"] = (
    google_test["Gross_Profit_Before_Ads"]
    - google_test["Ad_Spend"]
)

google_test["ROAS"] = (
    google_test["Expected_Revenue"]
    / google_test["Ad_Spend"]
)

google_test.round(2)

,Scenario,CPA,Ad_Spend,Expected_Orders,Expected_Revenue,Gross_Profit_Before_Ads,Profit_After_Ads,ROAS
0,Conservative,1100,5000,4.55,7117.64,5624.32,624.32,1.42
1,Base,800,5000,6.25,9786.75,7733.44,2733.44,1.96
2,Optimistic,550,5000,9.09,14235.27,11248.64,6248.64,2.85


### Google Ads Simulation

Google looks worth testing because historical Google traffic converted well.

In the base scenario, a 5,000 EGP Google Search budget could generate around 6 orders, 9,787 EGP in revenue and 2,733 EGP in gross profit after advertising.

Since Retroverse has never run Google Ads, these results are scenarios rather than expected guaranteed performance.

In [25]:
# Base-case Meta prospecting
meta_budget = 15000
meta_base_cpa = 700

meta_orders = meta_budget / meta_base_cpa
meta_revenue = meta_orders * 1565.88
meta_gross_profit = meta_orders * 1237.35
meta_profit_after_ads = meta_gross_profit - meta_budget


# Base-case Google Search
google_budget = 5000
google_base_cpa = 800

google_orders = google_budget / google_base_cpa
google_revenue = google_orders * 1565.88
google_gross_profit = google_orders * 1237.35
google_profit_after_ads = google_gross_profit - google_budget


# Base-case customer reactivation
reactivation_budget = 1000
reactivation_orders = 2
reactivation_revenue = reactivation_orders * 1565.88
reactivation_gross_profit = reactivation_orders * 1237.35
reactivation_profit_after_cost = (
    reactivation_gross_profit - reactivation_budget
)


combined_base = pd.DataFrame({
    "Strategy": [
        "Meta Prospecting",
        "Google Search",
        "Customer Reactivation"
    ],
    "Budget_EGP": [
        meta_budget,
        google_budget,
        reactivation_budget
    ],
    "Expected_Orders": [
        meta_orders,
        google_orders,
        reactivation_orders
    ],
    "Expected_Revenue": [
        meta_revenue,
        google_revenue,
        reactivation_revenue
    ],
    "Profit_After_Marketing": [
        meta_profit_after_ads,
        google_profit_after_ads,
        reactivation_profit_after_cost
    ]
})

combined_base.round(2)

,Strategy,Budget_EGP,Expected_Orders,Expected_Revenue,Profit_After_Marketing
0,Meta Prospecting,15000,21.43,33554.57,11514.64
1,Google Search,5000,6.25,9786.75,2733.44
2,Customer Reactivation,1000,2.00,3131.76,1474.70


In [26]:
print(
    "Modeled marketing spend:",
    combined_base["Budget_EGP"].sum(),
    "EGP"
)

print(
    "Expected additional orders:",
    round(combined_base["Expected_Orders"].sum(), 2)
)

print(
    "Expected additional revenue:",
    round(combined_base["Expected_Revenue"].sum(), 2),
    "EGP"
)

print(
    "Expected profit after marketing:",
    round(combined_base["Profit_After_Marketing"].sum(), 2),
    "EGP"
)

Modeled marketing spend: 21000 EGP
Expected additional orders: 29.68
Expected additional revenue: 46473.08 EGP
Expected profit after marketing: 15722.78 EGP


### Base Marketing Scenario

In the base scenario, the 21,000 EGP of marketing spend that can currently be modeled could generate around 30 additional orders and 46,473 EGP in additional revenue.

After estimated production and marketing costs, this would leave around 15,723 EGP in gross profit.

The remaining 9,000 EGP is not included in these predicted returns because 3,000 EGP is reserved for retargeting and 6,000 EGP will only be released after seeing which channel performs best.

In [27]:
campaign_months = 3

baseline_monthly_revenue = 1208.33

baseline_3m_revenue = (
    baseline_monthly_revenue * campaign_months
)

marketing_incremental_revenue = (
    combined_base["Expected_Revenue"].sum()
)

predicted_total_revenue = (
    baseline_3m_revenue
    + marketing_incremental_revenue
)

print(
    "3-month baseline revenue:",
    round(baseline_3m_revenue, 2),
    "EGP"
)

print(
    "Additional marketing revenue:",
    round(marketing_incremental_revenue, 2),
    "EGP"
)

print(
    "Predicted 3-month revenue:",
    round(predicted_total_revenue, 2),
    "EGP"
)

3-month baseline revenue: 3624.99 EGP
Additional marketing revenue: 46473.08 EGP
Predicted 3-month revenue: 50098.07 EGP


In [29]:
scenario_model = pd.DataFrame({
    "Scenario": [
        "Conservative",
        "Base",
        "Optimistic"
    ],

    "Meta_CPA": [
        1000,
        700,
        450
    ],

    "Google_CPA": [
        1100,
        800,
        550
    ],

    "Reactivation_Orders": [
        1,
        2,
        3
    ],

    "Monthly_Baseline": [
        604.17,
        1208.33,
        2957.78
    ]
})

# Marketing budgets
meta_budget = 15000
google_budget = 5000
reactivation_budget = 1000

# Historical-normal economics
avg_revenue_per_order = 1565.88
avg_gross_profit_per_order = 1237.35


# Expected orders
scenario_model["Meta_Orders"] = (
    meta_budget / scenario_model["Meta_CPA"]
)

scenario_model["Google_Orders"] = (
    google_budget / scenario_model["Google_CPA"]
)

scenario_model["Total_Additional_Orders"] = (
    scenario_model["Meta_Orders"]
    + scenario_model["Google_Orders"]
    + scenario_model["Reactivation_Orders"]
)


# Incremental revenue
scenario_model["Incremental_Revenue"] = (
    scenario_model["Total_Additional_Orders"]
    * avg_revenue_per_order
)


# 3-month baseline
scenario_model["Baseline_3M_Revenue"] = (
    scenario_model["Monthly_Baseline"] * 3
)


# Total predicted revenue
scenario_model["Predicted_3M_Revenue"] = (
    scenario_model["Baseline_3M_Revenue"]
    + scenario_model["Incremental_Revenue"]
)


# Incremental gross profit after modeled marketing
scenario_model["Incremental_Profit_After_Marketing"] = (
    scenario_model["Total_Additional_Orders"]
    * avg_gross_profit_per_order
    - meta_budget
    - google_budget
    - reactivation_budget
)

scenario_model.round(2)

,Scenario,Meta_CPA,Google_CPA,Reactivation_Orders,Monthly_Baseline,Meta_Orders,Google_Orders,Total_Additional_Orders,Incremental_Revenue,Baseline_3M_Revenue,Predicted_3M_Revenue,Incremental_Profit_After_Marketing
0,Conservative,1000,1100,1,604.17,15.00,4.55,20.55,32171.72,1812.51,33984.23,4421.92
1,Base,700,800,2,1208.33,21.43,6.25,29.68,46473.08,3624.99,50098.07,15722.78
2,Optimistic,450,550,3,2957.78,33.33,9.09,45.42,71128.91,8873.34,80002.25,35205.69


In [30]:
prediction_summary = scenario_model[
    [
        "Scenario",
        "Total_Additional_Orders",
        "Incremental_Revenue",
        "Baseline_3M_Revenue",
        "Predicted_3M_Revenue",
        "Incremental_Profit_After_Marketing"
    ]
].copy()

prediction_summary.round(2)

,Scenario,Total_Additional_Orders,Incremental_Revenue,Baseline_3M_Revenue,Predicted_3M_Revenue,Incremental_Profit_After_Marketing
0,Conservative,20.55,32171.72,1812.51,33984.23,4421.92
1,Base,29.68,46473.08,3624.99,50098.07,15722.78
2,Optimistic,45.42,71128.91,8873.34,80002.25,35205.69


### Scenario Explanation

I used three scenarios because Retroverse has never run paid Meta or Google Ads before, so the actual advertising performance is unknown.

**Conservative Scenario**

This assumes the ads are more expensive and sales remain weak.

- Meta CPA (Cost per Acquisition): 1,000 EGP
- Google CPA: 1,100 EGP
- 1 previous customer returns
- Baseline monthly revenue: 604 EGP

This represents a weaker campaign where it costs more to acquire customers.


**Base Scenario**

This is the middle scenario used as the main estimate.

- Meta CPA: 700 EGP
- Google CPA: 800 EGP
- 2 previous customers return
- Baseline monthly revenue: 1,208 EGP

This represents a reasonable result if the campaigns perform well but not exceptionally.


**Optimistic Scenario**

This assumes the ads perform very efficiently and normal sales activity improves.

- Meta CPA: 450 EGP
- Google CPA: 550 EGP
- 3 previous customers return
- Baseline monthly revenue: 2,958 EGP

This represents a strong campaign where customers are acquired at a lower cost.


These scenarios are not guaranteed predictions. They show how the expected results change depending on advertising performance and customer response.

### Marketing Prediction

The base scenario predicts around 30 additional orders and 46,473 EGP in additional revenue from the modeled marketing strategy.

Including expected sales without new marketing, total revenue over the three-month period could reach around 50,098 EGP.

The conservative scenario predicts around 33,984 EGP in total revenue, while the optimistic scenario reaches around 80,002 EGP.

The prediction is a range rather than a guaranteed result because Retroverse has no previous paid advertising data.

In [33]:
scale_rules = pd.DataFrame({
    "Channel": [
        "Meta",
        "Meta",
        "Meta",
        "Meta",
        "Google",
        "Google",
        "Google"
    ],

    "Performance": [
        "CPA <= 750 EGP",
        "CPA 751-1000 EGP",
        "CPA 1001-1237 EGP",
        "CPA > 1237 EGP",
        "CPA <= 800 EGP",
        "CPA 801-1237 EGP",
        "CPA > 1237 EGP"
    ],

    "Decision": [
        "Scale using reserve",
        "Keep testing before scaling",
        "Reduce spend and optimize",
        "Pause or change campaign",
        "Consider scaling using reserve",
        "Keep testing and optimize",
        "Pause or change campaign"
    ]
})

scale_rules

,Channel,Performance,Decision
0,Meta,CPA <= 750 EGP,Scale using reserve
1,Meta,CPA 751-1000 EGP,Keep testing before scaling
2,Meta,CPA 1001-1237 EGP,Reduce spend and optimize
3,Meta,CPA > 1237 EGP,Pause or change campaign
4,Google,CPA <= 800 EGP,Consider scaling using reserve
5,Google,CPA 801-1237 EGP,Keep testing and optimize
6,Google,CPA > 1237 EGP,Pause or change campaign


### Budget Scaling Rules

The 6,000 EGP reserve will only be spent after seeing the real campaign results.

If customer acquisition is cheap enough, the best-performing channel can receive more of the budget.

If the Cost per Acquisition gets close to the gross-profit break-even point of 1,237 EGP, spending should be reduced and the campaign should be improved before adding more money.

If it goes above 1,237 EGP, the campaign should be paused or changed.

In [34]:
current_poster_price = 2390
current_poster_cogs = 355

current_poster_gp = (
    current_poster_price - current_poster_cogs
)

current_poster_margin = (
    current_poster_gp / current_poster_price * 100
)

poster_ad_economics = pd.DataFrame({
    "CPA": [
        1000,
        700,
        450
    ]
})

poster_ad_economics["Selling_Price"] = current_poster_price
poster_ad_economics["COGS"] = current_poster_cogs
poster_ad_economics["Gross_Profit_Before_Ads"] = current_poster_gp

poster_ad_economics["Profit_After_Ads"] = (
    current_poster_gp
    - poster_ad_economics["CPA"]
)

poster_ad_economics["ROAS"] = (
    current_poster_price
    / poster_ad_economics["CPA"]
)

poster_ad_economics.round(2)

,CPA,Selling_Price,COGS,Gross_Profit_Before_Ads,Profit_After_Ads,ROAS
0,1000,2390,355,2035,1035,2.39
1,700,2390,355,2035,1335,3.41
2,450,2390,355,2035,1585,5.31


### Current Poster Economics

Regular posters currently sell for 2,390 EGP and cost around 355 EGP to produce.

This gives around 2,035 EGP in gross profit before advertising.

At a 700 EGP Cost per Acquisition, around 1,335 EGP would remain after production and advertising costs.

The current poster economics are stronger than the historical average order economics used in the main forecast, so the main prediction remains conservative.

In [35]:
final_budget = pd.DataFrame({
    "Strategy": [
        "Meta Prospecting",
        "Meta Retargeting",
        "Google Search Test",
        "Customer Reactivation",
        "Reserve for Best Channel"
    ],

    "Budget_EGP": [
        15000,
        3000,
        5000,
        1000,
        6000
    ],

    "Purpose": [
        "Acquire new customers",
        "Bring interested visitors back",
        "Capture high-intent search traffic",
        "Bring selected previous customers back",
        "Scale whichever channel performs best"
    ]
})

final_budget["Budget_%"] = (
    final_budget["Budget_EGP"]
    / final_budget["Budget_EGP"].sum()
    * 100
)

final_budget

,Strategy,Budget_EGP,Purpose,Budget_%
0,Meta Prospecting,15000,Acquire new customers,50.000000
1,Meta Retargeting,3000,Bring interested visitors back,10.000000
2,Google Search Test,5000,Capture high-intent search traffic,16.666667
3,Customer Reactivation,1000,Bring selected previous customers back,3.333333
4,Reserve for Best Channel,6000,Scale whichever channel performs best,20.000000


## Final Recommendation

Retroverse's biggest opportunity is acquiring new customers while improving retention.

Instagram has historically brought the most new customers, while Google traffic had the highest conversion rate. Custom Orders are the strongest product based on historical demand and profitability, while Custom Vinyl could be tested as a smaller growth opportunity.

The 30,000 EGP marketing budget should be split between Meta prospecting, Meta retargeting, Google Search, customer reactivation and a reserve that can be moved toward the best-performing channel.

The base scenario predicts around 50,098 EGP in total revenue over three months, with a scenario range of around 33,984 to 80,002 EGP.

These predictions are not guaranteed because Retroverse has no historical paid advertising data. The first campaigns should be treated as tests, and the remaining budget should only be scaled when the real Cost per Acquisition and Return on Ad Spend show that the campaign is profitable.

In [36]:
final_strategy = pd.DataFrame({
    "Area": [
        "Primary Customer",
        "Primary Geography",
        "Main Product",
        "Test Product",
        "Primary Acquisition Channel",
        "Secondary Acquisition Channel",
        "Retention Target",
        "Marketing Budget",
        "Base Predicted Revenue",
        "Predicted Revenue Range"
    ],

    "Recommendation": [
        "New customers",
        "Cairo, New Cairo, Sheikh Zayed / October",
        "Custom Orders / Posters",
        "Custom Vinyl",
        "Meta / Instagram",
        "Google Search",
        "10 medium/high priority previous customers",
        "30,000 EGP",
        "50,098 EGP over 3 months",
        "33,984 - 80,002 EGP over 3 months"
    ],

    "Evidence": [
        "Only 6.5% of identifiable customers purchased again",
        "Strongest areas based on historical orders and revenue",
        "Custom Orders had the strongest historical demand and high margins",
        "85% estimated margin but limited historical demand",
        "Instagram generated the most attributed orders and new customers",
        "Google had the highest historical conversion rate at 3.57%",
        "Customer propensity analysis identified the strongest reactivation targets",
        "Allocated across acquisition, retargeting, reactivation and performance reserve",
        "Base scenario using historical-normal order economics",
        "Conservative, base and optimistic scenario model"
    ]
})

final_strategy

,Area,Recommendation,Evidence
0,Primary Customer,New customers,Only 6.5% of identifiable customers purchased ...
1,Primary Geography,"Cairo, New Cairo, Sheikh Zayed / October",Strongest areas based on historical orders and...
2,Main Product,Custom Orders / Posters,Custom Orders had the strongest historical dem...
3,Test Product,Custom Vinyl,85% estimated margin but limited historical de...
4,Primary Acquisition Channel,Meta / Instagram,Instagram generated the most attributed orders...
5,Secondary Acquisition Channel,Google Search,Google had the highest historical conversion r...
6,Retention Target,10 medium/high priority previous customers,Customer propensity analysis identified the st...
7,Marketing Budget,"30,000 EGP","Allocated across acquisition, retargeting, rea..."
8,Base Predicted Revenue,"50,098 EGP over 3 months",Base scenario using historical-normal order ec...
9,Predicted Revenue Range,"33,984 - 80,002 EGP over 3 months","Conservative, base and optimistic scenario model"


In [37]:
final_budget.to_csv(
    CLEANED_DATA / "final_marketing_budget.csv",
    index=False
)

prediction_summary.to_csv(
    CLEANED_DATA / "marketing_prediction_scenarios.csv",
    index=False
)

scale_rules.to_csv(
    CLEANED_DATA / "marketing_scaling_rules.csv",
    index=False
)

final_strategy.to_csv(
    CLEANED_DATA / "final_marketing_strategy.csv",
    index=False
)

poster_ad_economics.to_csv(
    CLEANED_DATA / "current_poster_ad_economics.csv",
    index=False
)

print("Marketing outputs saved.")

Marketing outputs saved.


In [38]:
from pathlib import Path

powerbi_files = [
    "sales_orders.csv",
    "customer_rfm_segments.csv",
    "customer_segment_summary.csv",
    "customer_propensity.csv",
    "customer_priority_summary.csv",
    "final_marketing_budget.csv",
    "marketing_prediction_scenarios.csv",
    "marketing_scaling_rules.csv",
    "final_marketing_strategy.csv",
    "current_poster_ad_economics.csv"
]

for file in powerbi_files:
    path = CLEANED_DATA / file

    if path.exists():
        print("✓", file)
    else:
        print("MISSING:", file)

✓ sales_orders.csv
✓ customer_rfm_segments.csv
✓ customer_segment_summary.csv
✓ customer_propensity.csv
✓ customer_priority_summary.csv
✓ final_marketing_budget.csv
✓ marketing_prediction_scenarios.csv
✓ marketing_scaling_rules.csv
✓ final_marketing_strategy.csv
✓ current_poster_ad_economics.csv
